# 08 - Hill Saturation (Diminishing Returns)

## Objective

Marketing spend does **not** produce unlimited growth.

After a certain point, every additional ₹1 spent generates **less incremental sales**.

This phenomenon is called **Saturation** and is one of the core concepts of
Marketing Mix Modeling.

### Learning Goals

- Understand diminishing returns
- Implement the Hill Function
- Compare different alpha and gamma values
- Apply saturation to adstocked media
- Visualize response curves
- Prepare features for MMM



## Theory

The Hill function is commonly written as:

\[
f(x)=\frac{x^\alpha}{x^\alpha+\gamma^\alpha}
\]

Where:

- **x** = Adstocked media spend
- **α (alpha)** = Controls curve steepness
- **γ (gamma)** = Spend level where response reaches ~50%


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
DATA = ROOT/"data"/"processed"/"marketing_mix_adstock.csv"

df = pd.read_csv(DATA, parse_dates=["Week"])

media_cols = [
"Google_Search","Google_Display","Meta","Instagram",
"YouTube","TV","Radio","Influencer","Affiliate","Email"
]


## 1. Hill Function

In [ ]:

def hill_transform(x, alpha=1.5, gamma=None):
    x = np.asarray(x, dtype=float)
    if gamma is None:
        gamma = np.median(x)
    return (x**alpha) / ((x**alpha) + (gamma**alpha))


## 2. Visualize Different Alpha Values

In [ ]:

x = np.linspace(0,100,300)

plt.figure(figsize=(10,5))
for a in [0.5,1,2,3]:
    gamma=50
    y=(x**a)/((x**a)+(gamma**a))
    plt.plot(x,y,label=f"alpha={a}")

plt.title("Hill Saturation Curves")
plt.xlabel("Media Spend")
plt.ylabel("Normalized Response")
plt.legend()
plt.grid(True)
plt.show()


## 3. Visualize Gamma Effect

In [ ]:

plt.figure(figsize=(10,5))
for g in [20,40,60,80]:
    a=2
    y=(x**a)/((x**a)+(g**a))
    plt.plot(x,y,label=f"gamma={g}")

plt.title("Effect of Gamma")
plt.xlabel("Media Spend")
plt.ylabel("Response")
plt.legend()
plt.grid(True)
plt.show()


## 4. Apply Saturation After Adstock

In [ ]:

for col in media_cols:
    ad_col = col + "_Adstock"
    sat_col = col + "_Hill"

    df[sat_col] = hill_transform(df[ad_col], alpha=2)

df.filter(regex="Hill").head()


## 5. Compare Original vs Adstock vs Saturation

In [ ]:

plt.figure(figsize=(14,5))

plt.plot(df["Google_Search"],label="Original",alpha=.5)
plt.plot(df["Google_Search_Adstock"],label="Adstock")
plt.plot(df["Google_Search_Hill"],label="Hill Saturation")

plt.title("Google Search Transformation Pipeline")
plt.legend()
plt.show()


## 6. Correlation Comparison

In [ ]:

rows=[]

for col in media_cols:
    raw=df[[col,"Sales"]].corr().iloc[0,1]
    ad=df[[col+"_Adstock","Sales"]].corr().iloc[0,1]
    hill=df[[col+"_Hill","Sales"]].corr().iloc[0,1]

    rows.append([col,raw,ad,hill])

comparison=pd.DataFrame(
rows,
columns=["Channel","Raw","Adstock","Hill"]
)

display(comparison.sort_values("Hill",ascending=False))


## 7. Save Feature-Engineered Dataset

In [ ]:

OUT = ROOT/"data"/"processed"/"marketing_mix_hill.csv"
df.to_csv(OUT,index=False)
print("Saved:", OUT)


# Business Interpretation

## Recommended Transformation Order

1. Raw Media Spend
2. Adstock (carryover)
3. Hill Saturation (diminishing returns)
4. Feed transformed features into the MMM model

### Why?

- Adstock captures delayed advertising impact.
- Hill transformation captures diminishing returns.
- Together they provide a more realistic representation of media effectiveness.

## Typical Channel Behaviour

| Channel | Typical Saturation |
|----------|--------------------|
| Google Search | Medium |
| Meta | Medium |
| TV | Slow saturation |
| Radio | Medium |
| Email | Fast saturation |
| Influencer | Medium |
